# Diabetes 130 Hospitais — Pré-processamento dos Dados

**Objetivo:** Nesse notebook vamos preparar os dados brutos do dataset de diabetes para serem usados em modelos de machine learning. Basicamente vamos limpar, codificar e escalar os dados.

**O que vamos fazer (em ordem):**
1. Limpeza dos dados (remover colunas inúteis, tratar `?` e agrupar os diagnósticos ICD-9)
2. Preencher valores ausentes (NaN)
3. Tratar outliers usando IQR
4. Encoding (transformar categorias em números)
5. Escalonamento com RobustScaler
6. Seleção de features usando Mutual Information
7. Analisar se vale a pena aplicar PCA



## 1. Imports

In [ ]:
pip install ucimlrepo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
from ucimlrepo import fetch_ucirepo

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


## 2. Carregando os Dados e Dividindo em Treino/Teste

Aqui a gente carrega o dataset direto do UCI e já faz o split treino/teste. O `stratify=y` garante que a proporção das classes seja igual nos dois conjuntos, importante porque o dataset é desbalanceado.

Usamos `random_state=42` para que os resultados sejam reproduzíveis (qualquer um que rodar esse notebook vai ter o mesmo split).

In [ ]:
dataset = fetch_ucirepo(id=296)
X_raw = dataset.data.features
y_raw = dataset.data.targets

df = X_raw.copy()
df["readmitted"] = y_raw.iloc[:, 0].values

X = df.drop(columns=["readmitted"])
y = df["readmitted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")


/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Treino: (81412, 47) | Teste: (20354, 47)


In [ ]:
y.value_counts()

,count
readmitted,
NO,54864
>30,35545
<30,11357


## 3. Limpeza dos Dados

### 3.1 Removendo colunas inúteis, tratando `?` e agrupando os diagnósticos ICD-9


- `weight`: removemos porque 97% dos valores estavam ausentes. Sem dados suficientes pra o modelo aprender qualquer coisa com essa coluna.
- `examide` e `citoglipton`: Todos os dados tinham o valor `'No'`, ou seja, não há nenhuma variação. Um atributo assim não ajuda o modelo a distinguir nada.
- Os `'?'` foram substituídos por `NaN`, que é o formato padrão que o pandas e o sklearn entendem como valor ausente.
- `diag_1`, `diag_2` e `diag_3` tinham centenas de valores diferentes (códigos ICD-9). Agrupamos em 9 categorias clínicas (ex: "Circulatório", "Diabetes", "Respiratório") para reduzir a cardinalidade e dar um significado médico real às variáveis.

In [ ]:
COLS_DROP = ["weight", "examide", "citoglipton"]

ICD_GRUPOS = [
    ((390, 459), "Circulatory"),   ((785, 785), "Circulatory"),
    ((460, 519), "Respiratory"),   ((786, 786), "Respiratory"),
    ((520, 579), "Digestive"),     ((787, 787), "Digestive"),
    ((249, 250), "Diabetes"),
    ((800, 999), "Injury"),
    ((710, 739), "Musculoskeletal"),
    ((580, 629), "Genitourinary"), ((788, 788), "Genitourinary"),
    ((140, 239), "Neoplasms"),
]

def group_icd(code):
    """Classifica um código ICD-9 em categoria clínica principal."""
    s = str(code).strip()
    if not s or s in ('nan', '?'):
        return 'Other'
    if s[0] in ('V', 'E'):
        return 'Other'
    n = float(s.split('.')[0])
    for (lo, hi), label in ICD_GRUPOS:
        if lo <= n <= hi:
            return label
    return 'Other'



def clean_raw(df_in: pd.DataFrame) -> pd.DataFrame:
    """Remove colunas sem valor, converte '?' em NaN e agrupa ICD-9."""
    df = df_in.drop(columns=COLS_DROP, errors='ignore').copy()
    df = df.replace('?', np.nan)
    for col in ('diag_1', 'diag_2', 'diag_3'):
        if col in df.columns:
            df[col] = df[col].apply(group_icd)
    return df


X_train_clean = clean_raw(X_train)
X_test_clean  = clean_raw(X_test)

print(f"Shape após limpeza inicial: {X_train_clean.shape}")
print("\nNaN restantes no treino:")
nan_remaining = X_train_clean.isnull().sum()
print(nan_remaining[nan_remaining > 0].to_string())


Shape após limpeza inicial: (81412, 44)

NaN restantes no treino:
race                  1820
payer_code           32153
medical_specialty    39948
max_glu_serum        77162
A1Cresult            67767


### 3.2 Preenchendo os Valores Ausentes

Cada coluna com NaN recebeu um tratamento diferente, dependendo da quantidade de missings e do que faz sentido clinicamente:

| Coluna | % NaN | O que fizemos | Por quê |
|---|---|---|---|
| `max_glu_serum` | ~95% | Nova categoria: `'Não medido'` | A ausência do exame já é uma informação — paciente não monitorado |
| `A1Cresult` | ~83% | Nova categoria: `'Não medido'` | Mesmo raciocínio — HbA1c não coletada indica um perfil clínico diferente |
| `medical_specialty` | ~49% | Nova categoria: `'Desconhecido'` | Muitos ausentes; criar categoria preserva esse padrão |
| `payer_code` | ~40% | Nova categoria: `'Desconhecido'` | Mesmo raciocínio |
| `race` | ~2% | Moda do treino | Poucos ausentes, moda já é suficiente |
| `diag_1/2/3` | < 2% | Já tratado no `group_icd` | A função retorna `'Other'` quando o código é inválido ou ausente |

In [ ]:
# ── Aprender a moda de 'race' apenas no treino ──
race_mode = X_train_clean['race'].mode()[0]

# ── Manter apenas top-k categorias em colunas de alta cardinalidade ──
# Evita explosão de colunas no OHE e melhora generalização
TOP_PAYER   = 10
TOP_MEDICAL = 20

top_payer_cats   = X_train_clean['payer_code'].value_counts().nlargest(TOP_PAYER).index.tolist()
top_medical_cats = X_train_clean['medical_specialty'].value_counts().nlargest(TOP_MEDICAL).index.tolist()


def fill_and_reduce(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Imputa NaN e reduz cardinalidade em colunas de alta missingness.
    Todos os parâmetros (moda, top-cats) foram aprendidos no treino.
    """
    df = df_in.copy()

    # Imputar com categoria informativa
    df['max_glu_serum']    = df['max_glu_serum'].fillna('Não medido')
    df['A1Cresult']        = df['A1Cresult'].fillna('Não medido')
    df['medical_specialty']= df['medical_specialty'].fillna('Desconhecido')
    df['payer_code']       = df['payer_code'].fillna('Desconhecido')

    # Imputar race com moda do treino
    df['race'] = df['race'].fillna(race_mode)

    # Reduzir cardinalidade: categorias fora do top-k → 'Other'
    df['payer_code']        = df['payer_code'].apply(
        lambda x: x if x in top_payer_cats else 'Other')
    df['medical_specialty'] = df['medical_specialty'].apply(
        lambda x: x if x in top_medical_cats else 'Other')

    return df


X_train_filled = fill_and_reduce(X_train_clean)
X_test_filled  = fill_and_reduce(X_test_clean)

remaining = X_train_filled.isnull().sum()
if remaining.sum() == 0:
    print("Nenhum NaN restante no treino ✓")
else:
    print(remaining[remaining > 0])


Nenhum NaN restante no treino ✓


### 3.3 Tratamento de Outliers com IQR Clipping

**Como funciona:** A técnica de Winsorização com IQR calcula os limites inferior e superior (Q1 - 1.5×IQR e Q3 + 1.5×IQR) e "recorta" os valores que estão fora desse intervalo, substituindo-os pelo limite mais próximo.

**Por que não usamos clustering (DBSCAN etc.) pra detectar outliers?** Com 80 mil linhas ficaria muito pesado computacionalmente, e seria difícil aplicar o resultado de forma consistente no conjunto de teste sem causar leakage.

**Por que clipping em vez de remover as linhas?** Porque internações muito longas ou com muitos procedimentos podem ser casos clínicos reais e legítimos — remover esses pacientes seria jogar fora informação válida.

Os limites do IQR são calculados só no treino e depois aplicados nos dois conjuntos.

In [ ]:
NUMERIC_COLS = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]


def compute_iqr_bounds(df: pd.DataFrame, cols: list, factor: float = 1.5) -> dict:
    """Calcula limites de clipping IQR. Chamado apenas no treino."""
    bounds = {}
    for col in cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - factor * iqr, q3 + factor * iqr)
    return bounds


def clip_outliers(df_in: pd.DataFrame, bounds: dict) -> pd.DataFrame:
    """Aplica clipping com limites pré-calculados (sem leakage no teste)."""
    df = df_in.copy()
    for col, (lo, hi) in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=lo, upper=hi)
    return df


# Aprender limites APENAS no treino
iqr_bounds = compute_iqr_bounds(X_train_filled, NUMERIC_COLS)

print(f"{'Coluna':<30} {'Outliers (treino)':>20} {'%':>8}")
print("-" * 62)
for col, (lo, hi) in iqr_bounds.items():
    n_out = ((X_train_filled[col] < lo) | (X_train_filled[col] > hi)).sum()
    print(f"{col:<30} {n_out:>20,d} {n_out/len(X_train_filled)*100:>7.2f}%")

X_train_clipped = clip_outliers(X_train_filled, iqr_bounds)
X_test_clipped  = clip_outliers(X_test_filled, iqr_bounds)


Coluna                            Outliers (treino)        %
--------------------------------------------------------------
time_in_hospital                              1,784    2.19%
num_lab_procedures                              116    0.14%
num_procedures                                3,989    4.90%
num_medications                               2,053    2.52%
number_outpatient                            13,392   16.45%
number_emergency                              9,066   11.14%
number_inpatient                              5,605    6.88%
number_diagnoses                                218    0.27%


## 4. Encoding das Variáveis Categóricas

Como os modelos de ML só trabalham com números, precisamos converter as variáveis categóricas. A estratégia usada variou de acordo com o tipo de cada variável:

| Tipo | Variáveis | Técnica | Por quê |
|---|---|---|---|
| **Ordinal com ordem clara** | `age` | `OrdinalEncoder` com categorias fixas | Faixas etárias têm uma ordem natural bem definida |
| **Ordinal de medicamento** | `metformin`, `insulin`, etc. (21 colunas) | `OrdinalEncoder` (No=0, Down=1, Steady=2, Up=3) | Há uma escala de intensidade de uso |
| **Binárias** | `gender`, `change`, `diabetesMed` | Mapeamento manual 0/1 | Simples e direto, sem criar colunas extras desnecessárias |
| **Nominais (baixa cardinalidade)** | `race`, `max_glu_serum`, `A1Cresult`, `diag_1/2/3` | `OneHotEncoder(drop='first')` | Sem hierarquia entre as categorias; `drop='first'` evita multicolinearidade |
| **IDs categóricos** | `admission_type_id`, `discharge_disposition_id`, `admission_source_id` | `OneHotEncoder(drop='first')` | São categorias, não valores numéricos contínuos |
| **Nominais (alta cardinalidade)** | `payer_code` (top-10+Other), `medical_specialty` (top-20+Other) | `OneHotEncoder(drop='first')` | Cardinalidade já reduzida na etapa anterior |

> Para as variáveis numéricas, usamos o `RobustScaler`, que escala usando mediana e IQR em vez de média e desvio padrão — isso o torna mais resistente a outliers que ainda possam existir após o clipping.

In [ ]:
# ── Encoding manual das binárias ──────────────────────────────────────────
BINARY_MAP = {
    'gender':      {'Female': 0, 'Male': 1},
    'change':      {'No': 0, 'Ch': 1},
    'diabetesMed': {'No': 0, 'Yes': 1},
}

def encode_binary_cols(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    for col, mapping in BINARY_MAP.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(0).astype(int)
    # IDs categóricos → string para o OHE não interpretá-los como contínuos
    for col in ('admission_type_id', 'discharge_disposition_id', 'admission_source_id'):
        if col in df.columns:
            df[col] = df[col].astype(str)
    return df


X_train_enc = encode_binary_cols(X_train_clipped)
X_test_enc  = encode_binary_cols(X_test_clipped)

# ── Definição dos grupos de colunas ───────────────────────────────────────────
AGE_ORDER = [['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
              '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']]

MED_COLS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]
MED_ORDER = [['No', 'Down', 'Steady', 'Up']] * len(MED_COLS)

BINARY_COLS  = list(BINARY_MAP.keys())

NOMINAL_COLS = [
    'race', 'max_glu_serum', 'A1Cresult',
    'diag_1', 'diag_2', 'diag_3',
    'payer_code', 'medical_specialty',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
]

# ── ColumnTransformer ─────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        # Numérico → RobustScaler
        ('num',
         RobustScaler(),
         NUMERIC_COLS),

        # Age ordinal
        ('age_ord',
         OrdinalEncoder(categories=AGE_ORDER,
                        handle_unknown='use_encoded_value', unknown_value=-1),
         ['age']),

        # Medicamentos ordinais
        ('med_ord',
         OrdinalEncoder(categories=MED_ORDER,
                        handle_unknown='use_encoded_value', unknown_value=-1),
         MED_COLS),

        # Binárias → passthrough (já mapeadas)
        ('bin', 'passthrough', BINARY_COLS),

        # Nominais → OHE com drop='first'
        ('ohe',
         OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
         NOMINAL_COLS),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

# fit no treino → transform em ambos
X_train_transformed = preprocessor.fit_transform(X_train_enc)
X_test_transformed  = preprocessor.transform(X_test_enc)

print(f"Shape treino após encoding + scaling: {X_train_transformed.shape}")
print(f"Shape teste  após encoding + scaling: {X_test_transformed.shape}")


Shape treino após encoding + scaling: (81412, 145)
Shape teste  após encoding + scaling: (20354, 145)


In [ ]:
# ── Nomes das features pós-transformação ─────────────────────────────────────
ohe_names = preprocessor.named_transformers_['ohe']            .get_feature_names_out(NOMINAL_COLS).tolist()

all_feature_names = (
    NUMERIC_COLS
    + ['age_encoded']
    + [f'{c}_enc' for c in MED_COLS]
    + BINARY_COLS
    + ohe_names
)

X_train_df = pd.DataFrame(X_train_transformed,
                           columns=all_feature_names)
X_test_df  = pd.DataFrame(X_test_transformed,
                           columns=all_feature_names)

print(f"Total de features após encoding: {len(all_feature_names)}")
print("\nPrimeiras 5 colunas:", all_feature_names[:5])
print("Últimas  5 colunas:", all_feature_names[-5:])


Total de features após encoding: 145

Primeiras 5 colunas: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient']
Últimas  5 colunas: ['admission_source_id_5', 'admission_source_id_6', 'admission_source_id_7', 'admission_source_id_8', 'admission_source_id_9']


## 5. Seleção de Features com Mutual Information

Depois do encoding ficamos com 145 features, o que é bastante. Para reduzir, usamos o `SelectKBest` com `mutual_info_classif`.

**Por que Mutual Information e não chi² ou correlação de Pearson?**
- Captura relações **não-lineares** entre as features e o target, o que é comum em dados clínicos.
- Funciona bem com dados mistos (contínuos + categóricos já codificados).
- Suporta múltiplas classes no target — o nosso tem 3: `NO`, `<30` e `>30`.

**Critério de corte:** mantivemos as features com MI Score maior ou igual à mediana de todos os scores. Isso descarta a metade menos informativa e mantém as 73 mais relevantes.

In [ ]:
# Avaliar MI Score de todas as features (fit apenas no treino)
mi_selector = SelectKBest(score_func=mutual_info_classif, k='all')
mi_selector.fit(X_train_transformed, y_train)

mi_scores = pd.Series(mi_selector.scores_, index=all_feature_names)              .sort_values(ascending=False)

# ── Visualização — Top 30 ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 9))
mi_scores.head(30).sort_values().plot(
    kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.axvline(mi_scores.median(), color='red', linestyle='--',
           linewidth=1.5, label=f'Mediana ({mi_scores.median():.4f})')
ax.set_title("Top 30 Features — Mutual Information Score (X_train)")
ax.set_xlabel("MI Score")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Aplicar corte pela mediana ────────────────────────────────────────────────
mi_threshold  = mi_scores.median()
selected_feats = mi_scores[mi_scores >= mi_threshold].index.tolist()

print(f"Features totais:       {len(all_feature_names)}")
print(f"Features selecionadas: {len(selected_feats)}  (MI >= {mi_threshold:.4f})")
print(f"Features descartadas:  {len(all_feature_names) - len(selected_feats)}")
print()
print("Top 10 features mais relevantes:")
print(mi_scores.head(10).to_string())

X_train_selected = X_train_df[selected_feats].values
X_test_selected  = X_test_df[selected_feats].values


Features totais:       145
Features selecionadas: 73  (MI >= 0.0009)
Features descartadas:  72

Top 10 features mais relevantes:
number_inpatient               0.032564
discharge_disposition_id_11    0.012094
max_glu_serum_Não medido       0.010467
number_diagnoses               0.010135
race_Caucasian                 0.009682
diabetesMed                    0.007323
age_encoded                    0.006516
admission_source_id_7          0.006463
num_medications                0.006322
change                         0.006191


## 6. Análise PCA — Vale a Pena Reduzir a Dimensionalidade?

O PCA transforma as features em componentes que capturam a maior parte da variância dos dados. A ideia aqui é verificar se conseguimos representar os dados com menos dimensões sem perder muita informação.

**Critério que usamos:** só aplicamos PCA se conseguirmos uma redução de mais de 30% das features mantendo pelo menos 90% da variância. Se a redução for pequena, não vale o custo — o PCA dificulta a interpretação dos resultados porque as features originais deixam de existir.

In [ ]:
# Explorar variância cumulativa do PCA
n_components_explore = min(len(selected_feats), X_train_selected.shape[0], 150)
pca_explorer = PCA(n_components=n_components_explore, random_state=42)
pca_explorer.fit(X_train_selected)

cumvar = np.cumsum(pca_explorer.explained_variance_ratio_)
n_90   = int(np.argmax(cumvar >= 0.90)) + 1
n_95   = int(np.argmax(cumvar >= 0.95)) + 1

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar, linewidth=2, color='steelblue')
ax.fill_between(range(1, len(cumvar) + 1), cumvar, alpha=0.15, color='steelblue')
ax.axhline(0.90, color='red',    linestyle='--', label=f'90% variância → {n_90} componentes')
ax.axhline(0.95, color='orange', linestyle='--', label=f'95% variância → {n_95} componentes')
ax.axvline(n_90, color='red',    linestyle=':',  alpha=0.7)
ax.axvline(n_95, color='orange', linestyle=':',  alpha=0.7)
ax.set_xlabel("Número de componentes")
ax.set_ylabel("Variância acumulada explicada")
ax.set_title("PCA — Curva de Variância Acumulada (X_train selecionado)")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Dimensões originais (pós-seleção): {len(selected_feats)}")
print(f"Componentes para 90% de variância: {n_90}")
print(f"Componentes para 95% de variância: {n_95}")
reducao_90 = (1 - n_90 / len(selected_feats)) * 100
print(f"Redução para 90%: {reducao_90:.1f}%")


Dimensões originais (pós-seleção): 73
Componentes para 90% de variância: 23
Componentes para 95% de variância: 30
Redução para 90%: 68.5%


In [ ]:
# ── Decisão automática sobre PCA ─────────────────────────────────────────────
REDUCAO_MINIMA_PCT = 30.0   # Só aplica PCA se reduzir mais de 30% das features

if reducao_90 >= REDUCAO_MINIMA_PCT:
    print(f"✓ PCA APLICADO: {len(selected_feats)} → {n_90} componentes "
          f"({reducao_90:.1f}% de redução) mantendo 90% da variância.")
    pca_final = PCA(n_components=n_90, random_state=42)
    X_train_final = pca_final.fit_transform(X_train_selected)
    X_test_final  = pca_final.transform(X_test_selected)
    final_col_names = [f'PC{i+1}' for i in range(n_90)]
else:
    print(f"✗ PCA NÃO APLICADO: redução seria apenas {reducao_90:.1f}% "
          f"({len(selected_feats)} → {n_90}). "
          f"Custo em interpretabilidade não justificado.")
    pca_final       = None
    X_train_final   = X_train_selected
    X_test_final    = X_test_selected
    final_col_names = selected_feats

X_train_final_df = pd.DataFrame(X_train_final, columns=final_col_names)
X_test_final_df  = pd.DataFrame(X_test_final,  columns=final_col_names)

print(f"\nDataset final — TREINO: {X_train_final_df.shape}")
print(f"Dataset final — TESTE:  {X_test_final_df.shape}")


✓ PCA APLICADO: 73 → 23 componentes (68.5% de redução) mantendo 90% da variância.

Dataset final — TREINO: (81412, 23)
Dataset final — TESTE:  (20354, 23)


## 7. Resumo do que foi feito

```
Dados brutos (101.766 × 47)
  │
  ├─ train_test_split (stratify, random_state=42)
  │   ├─ X_train  81.412 × 47
  │   └─ X_test   20.354 × 47
  │
  ▼ [tudo aprendido no treino e só aplicado nos dois]
  ├─ 1. Limpeza
  │   ├─ Drop: weight, examide, citoglipton
  │   ├─ '?' → NaN
  │   └─ diag_1/2/3: ICD-9 → 9 categorias clínicas
  │
  ├─ 2. Imputação
  │   ├─ max_glu_serum / A1Cresult → 'Não medido'
  │   ├─ medical_specialty / payer_code → 'Desconhecido' + top-k categorias
  │   └─ race → moda do treino
  │
  ├─ 3. Outliers — IQR Clipping (fator 1.5, limites calculados no treino)
  │
  ├─ 4. Encoding + Scaling (ColumnTransformer)
  │   ├─ RobustScaler           → 8 colunas numéricas
  │   ├─ OrdinalEncoder         → age (10 faixas) + 21 colunas de medicamentos
  │   ├─ Mapeamento 0/1         → gender, change, diabetesMed
  │   └─ OneHotEncoder(drop=1st)→ race, glu, A1C, diag, payer, specialty, IDs
  │
  ├─ 5. Seleção — SelectKBest(mutual_info_classif) com corte pela mediana
  │
  └─ 6. PCA (só se redução > 30%)
```

**Datasets prontos para a modelagem:**

| Objeto | Descrição |
|---|---|
| `X_train_final_df` | Features de treino prontas |
| `X_test_final_df` | Features de teste prontas |
| `y_train` | Labels de treino |
| `y_test` | Labels de teste |
| `preprocessor` | Pipeline fit no treino (para usar em novos dados) |
| `pca_final` | PCA fit, ou `None` se não foi aplicado |

In [ ]:
#@title Integracao Arvores de Decisao

# Importar as bibliotecas necessárias
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from sklearn import tree

# Carregar a base de dados pré-processada
X_treinamento = X_train_final_df
X_teste = X_test_final_df
y_treinamento = y_train
y_teste = y_test

# Etapa 1: Entendimento do negócio e análise dos dados
# (Esta etapa já foi realizada no pré-processamento)

# Separar os atributos de entrada (X) e as classes (y) - já feito com X_train_final_df e y_train
X = pd.concat([X_treinamento, X_teste])
y = pd.concat([y_treinamento, y_teste])

# Etapa 2: Preparação dos dados
# Dividir a base de dados em treinamento e validacao (já feito no pré-processamento)

# Etapa 3: Modelagem

# Criar o modelo de árvore de decisão
modelo = DecisionTreeClassifier()

# Etapa 4: Avaliação do modelo

# Realizar a validação cruzada com K-fold estratificado de 10 vezes
skf = StratifiedKFold(n_splits=10)
resultados_treinamento = []
resultados_validacao = []

# Converter DataFrames para arrays numpy para skf.split, se necessário
# (skf.split funciona com DataFrames, mas iloc pode precisar de reset_index se as colunas não forem numéricas)
# X_treinamento e X_teste são DataFrames, y_treinamento e y_teste são Series

for indice_treinamento, indice_validacao in skf.split(X_treinamento, y_treinamento):
    X_treino, X_validacao = X_treinamento.iloc[indice_treinamento], X_treinamento.iloc[indice_validacao]
    y_treino, y_validacao = y_treinamento.iloc[indice_treinamento], y_treinamento.iloc[indice_validacao]

    modelo.fit(X_treino, y_treino)
    resultados_treinamento.append(modelo.score(X_treino, y_treino))
    resultados_validacao.append(modelo.score(X_validacao, y_validacao))

# Plotar a evolução do desempenho de treinamento e validação
plt.plot(range(1, 11), resultados_treinamento, label='Treinamento')
plt.plot(range(1, 11), resultados_validacao, label='Validação')
plt.xlabel('Fold')
plt.ylabel('Acurácia')
plt.title('Evolução do Desempenho de Treinamento e Validação')
plt.legend()
plt.show()

# Etapa 5: Construção do modelo final

# Treinar o modelo final usando todos os dados de treinamento
modelo_final = DecisionTreeClassifier()
modelo_final.fit(X_treinamento, y_treinamento)

# Etapa 6: Avaliação do modelo final

# Fazer a classificação dos dados de teste
y_pred = modelo_final.predict(X_teste)

# Apresentar a matriz de confusão
matriz_confusao = confusion_matrix(y_teste, y_pred)
print("Matriz de Confusão:")
print(matriz_confusao)

# Apresentar as métricas de F1, acurácia e recall
report = classification_report(y_teste, y_pred)
print("Métricas:")
print(report)

# Apresentar o relatório de classificação
relatorio_classificacao = classification_report(y_teste, y_pred)
print("Relatório de Classificação:")
print(relatorio_classificacao)

# Plotar a árvore de decisão
plt.figure(figsize=(12, 8))
# Ajustar feature_names e class_names para o novo dataset
tree.plot_tree(modelo_final, feature_names=X_treinamento.columns, class_names=modelo_final.classes_.astype(str), filled=True)
plt.show()

Matriz de Confusão:
[[ 317  886 1069]
 [ 906 2689 3514]
 [1171 3620 6182]]
Métricas:
              precision    recall  f1-score   support

         <30       0.13      0.14      0.14      2272
         >30       0.37      0.38      0.38      7109
          NO       0.57      0.56      0.57     10973

    accuracy                           0.45     20354
   macro avg       0.36      0.36      0.36     20354
weighted avg       0.45      0.45      0.45     20354

Relatório de Classificação:
              precision    recall  f1-score   support

         <30       0.13      0.14      0.14      2272
         >30       0.37      0.38      0.38      7109
          NO       0.57      0.56      0.57     10973

    accuracy                           0.45     20354
   macro avg       0.36      0.36      0.36     20354
weighted avg       0.45      0.45      0.45     20354



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Definir as cores usadas nos gráficos, se não estiverem definidas
# Exemplo de definição:
COR_A = '#e6550d'
COR_B = '#3182bd'
TEXTO = 'white'

#@title Integracao KNN

# Importar as bibliotecas necessárias
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("Iniciando análise k-NN com o dataset pré-processado do usuário...")

# Carregar os dados pré-processados do usuário
X_train_user = X_train_final_df
X_test_user = X_test_final_df
y_train_user = y_train
y_test_user = y_test

print(f"Dataset do usuário — Treino: {X_train_user.shape} | Teste: {X_test_user.shape}")

# Obter nomes das classes para o dataset do usuário
# Garantir que os nomes das classes estejam em ordem para a matriz de confusão
user_class_names = np.unique(y_train_user.values) # Convert to numpy array to ensure correct unique values if series has duplicates/order issues

# Curva de acurácia vs k para o dataset do usuário
k_vals_user = list(range(1, 21)) # Usar a mesma faixa de k
acc_train_user = []
acc_test_user = []

for k in k_vals_user:
    # Não aplicar StandardScaler novamente, pois os dados já são o resultado do PCA
    clf_user = KNeighborsClassifier(n_neighbors=k).fit(X_train_user, y_train_user)
    acc_train_user.append(clf_user.score(X_train_user, y_train_user))
    acc_test_user.append(clf_user.score(X_test_user, y_test_user))

# Determinar o melhor k baseado na acurácia do conjunto de teste
best_k_idx_user = np.argmax(acc_test_user)
best_k_val_user = k_vals_user[best_k_idx_user]

# Treinar o modelo final com o melhor k
clf_best_user = KNeighborsClassifier(n_neighbors=best_k_val_user).fit(X_train_user, y_train_user)
y_pred_user = clf_best_user.predict(X_test_user)

# ─────────────────────────────────────────────────────────────────────
# Visualização dos Resultados para o Dataset do Usuário
# ─────────────────────────────────────────────────────────────────────

fig_user = plt.figure(figsize=(17, 5))
gs_user = gridspec.GridSpec(1, 3, figure=fig_user, wspace=0.38)
fig_user.suptitle('k-NN — Análise com Dataset Pré-processado do Usuário',
                  fontsize=14, fontweight='bold', color='white', y=1.02)

# Painel 1 — Curva de acurácia vs k
ax_user1 = fig_user.add_subplot(gs_user[0, 0])
ax_user1.plot(k_vals_user, acc_train_user, 'o-', color=COR_B, lw=2.5, ms=6, label='Treino')
ax_user1.plot(k_vals_user, acc_test_user,  's--', color=COR_A, lw=2.5, ms=6, label='Teste', alpha=0.85)
ax_user1.fill_between(k_vals_user, acc_train_user, acc_test_user,
                 where=[tr > te for tr, te in zip(acc_train_user, acc_test_user)],
                 alpha=0.15, color=COR_A, label='Gap overfitting')
ax_user1.axvline(x=best_k_val_user, color='white', lw=1.5, ls=':', label=f'Melhor k={best_k_val_user}')
ax_user1.set_xlabel('Valor de k'); ax_user1.set_ylabel('Acurácia')
ax_user1.set_ylim(0.5, 1.05); ax_user1.set_xticks(k_vals_user[::2])
ax_user1.set_title(f'Acurácia vs k para Dataset do Usuário', fontweight='bold', color='white')
ax_user1.legend(facecolor='#1B2A4A', labelcolor='white', framealpha=0.9, fontsize=9)
ax_user1.grid(True)

# Painel 2 — Matriz de Confusão (melhor modelo)
cm_user = confusion_matrix(y_test_user, y_pred_user, labels=user_class_names)

ax_user2 = fig_user.add_subplot(gs_user[0, 1])
im_user = ax_user2.imshow(cm_user, cmap='Blues')
ax_user2.set_xticks(range(len(user_class_names))); ax_user2.set_yticks(range(len(user_class_names)))
ax_user2.set_xticklabels(user_class_names, rotation=20, ha='right')
ax_user2.set_yticklabels(user_class_names)
ax_user2.set_xlabel('Predito'); ax_user2.set_ylabel('Real')
ax_user2.set_title(f'Matriz de Confusão (k={best_k_val_user})',
                  fontweight='bold', color='white')
for i in range(len(user_class_names)):
    for j in range(len(user_class_names)):
        ax_user2.text(j, i, str(cm_user[i, j]),
                     ha='center', va='center',
                     color='white' if cm_user[i, j] > cm_user.max()/2 else '#1B2A4A',
                     fontsize=14, fontweight='bold')

# Painel 3 — Relatório de Classificação (Texto)
ax_user3 = fig_user.add_subplot(gs_user[0, 2])
ax_user3.axis('off') # Esconder eixos para exibir texto
ax_user3.set_title('Relatório de Classificação', fontweight='bold', color='white')

report_user_dict = classification_report(y_test_user, y_pred_user, target_names=user_class_names, output_dict=True)
report_str = """
{: <15} {: >8} {: >8} {: >8} {: >8}
""".format('Classe', 'Precision', 'Recall', 'F1-Score', 'Support')
report_str += "-" * 50 + "\n"

for label in user_class_names:
    metrics = report_user_dict.get(label, {})
    report_str += "{: <15} {: >8.2f} {: >8.2f} {: >8.2f} {: >8.0f}\n".format(
        label,
        metrics.get('precision', 0.0),
        metrics.get('recall', 0.0),
        metrics.get('f1-score', 0.0),
        metrics.get('support', 0)
    )

report_str += "\n" + "-" * 50 + "\n"
# Adicionar macro avg e weighted avg
for avg_type in ['accuracy', 'macro avg', 'weighted avg']:
    metrics = report_user_dict.get(avg_type, {})
    if isinstance(metrics, dict):
        report_str += "{: <15} {: >8.2f} {: >8.2f} {: >8.2f} {: >8.0f}\n".format(
            avg_type,
            metrics.get('precision', 0.0),
            metrics.get('recall', 0.0),
            metrics.get('f1-score', 0.0),
            metrics.get('support', 0)
        )
    else: # accuracy is not a dict
        report_str += "{: <15} {: >8.2f} {: >8} {: >8} {: >8.0f}\n".format(
            avg_type,
            metrics, '', '', report_user_dict['macro avg']['support'] # Total support for accuracy
        )

ax_user3.text(0, 1, report_str,
              transform=ax_user3.transAxes,
              fontsize=9, va='top', ha='left', color=TEXTO, family='monospace')

plt.tight_layout()
plt.show()

print("\n✅ k-NN com Dataset do Usuário concluído.")
print(f"   Melhor k encontrado: {best_k_val_user}")
print(f"   Acurácia no Treino (melhor k): {clf_best_user.score(X_train_user, y_train_user):.1%}")
print(f"   Acurácia no Teste (melhor k): {clf_best_user.score(X_test_user, y_test_user):.1%}")

Iniciando análise k-NN com o dataset pré-processado do usuário...
Dataset do usuário — Treino: (81412, 23) | Teste: (20354, 23)


/tmp/ipykernel_18139/2269224515.py:133: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



✅ k-NN com Dataset do Usuário concluído.
   Melhor k encontrado: 20
   Acurácia no Treino (melhor k): 59.8%
   Acurácia no Teste (melhor k): 55.1%


In [ ]:
#@title Integracao LVQ1

print("Iniciando integração LVQ1 com o dataset pré-processado do usuário...")

# ─── Implementação do LVQ1 ────────────────────────────────────────
class LVQ1:
    """
    Learning Vector Quantization 1 — Kohonen (1986)

    Parâmetros
    ----------
    n_prototypes : int
        Número de protótipos por classe
    learning_rate : float
        Taxa de aprendizado inicial α₀
    n_epochs : int
        Número de épocas de treinamento
    decay : float
        Fator de decaimento da taxa (α(t) = α₀ × decay^t)
    random_state : int
        Semente para reprodutibilidade
    """
    def __init__(self, n_prototypes=2, learning_rate=0.3,
                 n_epochs=50, decay=0.95, random_state=42):
        self.n_proto      = n_prototypes
        self.lr0          = learning_rate
        self.n_epochs     = n_epochs
        self.decay        = decay
        self.rng          = np.random.RandomState(random_state)
        self.history      = []   # histórico de acurácia por época

    def fit(self, X, y, X_val=None, y_val=None):
        classes = np.unique(y)
        self.classes_ = classes
        self.prototypes_    = []
        self.proto_labels_  = []

        # Inicialização: primeiros n_proto exemplos de cada classe
        for c in classes:
            idx = np.where(y == c)[0]
            chosen = self.rng.choice(idx, size=self.n_proto, replace=False)
            for i in chosen:
                self.prototypes_.append(X[i].copy().astype(float))
                self.proto_labels_.append(c)

        self.prototypes_   = np.array(self.prototypes_)
        self.proto_labels_ = np.array(self.proto_labels_)

        # Snapshots dos protótipos para animação
        self.proto_snapshots = [self.prototypes_.copy()]

        lr = self.lr0
        for epoch in range(self.n_epochs):
            idx_perm = self.rng.permutation(len(X))
            for i in idx_perm:
                x, label = X[i], y[i]
                # Passo 1: encontra o protótipo vencedor (mais próximo)
                dists  = np.linalg.norm(self.prototypes_ - x, axis=1)
                winner = np.argmin(dists)
                # Passo 2: atualiza conforme acerto/erro
                if self.proto_labels_[winner] == label:
                    # Acerto — aproxima
                    self.prototypes_[winner] += lr * (x - self.prototypes_[winner])
                else:
                    # Erro — afasta
                    self.prototypes_[winner] -= lr * (x - self.prototypes_[winner])
            # Taxa de aprendizado decai a cada época
            lr *= self.decay
            # Registra acurácia e snapshot
            acc = self.score(X, y)
            self.history.append({
                'epoch': epoch + 1,
                'lr': lr,
                'acc_treino': acc,
                'acc_val': self.score(X_val, y_val) if X_val is not None else None
            })
            self.proto_snapshots.append(self.prototypes_.copy())

        return self

    def predict(self, X):
        preds = []
        for x in X:
            dists = np.linalg.norm(self.prototypes_ - x, axis=1)
            preds.append(self.proto_labels_[np.argmin(dists)])
        return np.array(preds)

    def score(self, X, y):
        return float(np.mean(self.predict(X) == y))


print("✅ LVQ1 implementado.")

# Carregar os dados pré-processados do usuário
X_train_user_lvq = X_train_final_df.values  # Convert to numpy array for LVQ
X_test_user_lvq = X_test_final_df.values    # Convert to numpy array for LVQ
y_train_user_lvq = y_train.values          # Convert to numpy array for LVQ
y_test_user_lvq = y_test.values            # Convert to numpy array for LVQ

print(f"Dataset do usuário — Treino: {X_train_user_lvq.shape} | Teste: {X_test_user_lvq.shape}")

# Obter nomes das classes para o dataset do usuário
user_class_names = np.unique(y_train_user_lvq)

# Curva de acurácia vs número de protótipos
n_prototypes_range = list(range(1, 6)) # Testar de 1 a 5 protótipos por classe
acc_train_lvq = []
acc_test_lvq = []

for n_p in n_prototypes_range:
    lvq_model = LVQ1(n_prototypes=n_p, learning_rate=0.1, n_epochs=100, random_state=42)
    lvq_model.fit(X_train_user_lvq, y_train_user_lvq, X_val=X_test_user_lvq, y_val=y_test_user_lvq)
    acc_train_lvq.append(lvq_model.score(X_train_user_lvq, y_train_user_lvq))
    acc_test_lvq.append(lvq_model.score(X_test_user_lvq, y_test_user_lvq))

# Determinar o melhor n_prototypes baseado na acurácia do conjunto de teste
best_n_prototypes_idx = np.argmax(acc_test_lvq)
best_n_prototypes_val = n_prototypes_range[best_n_prototypes_idx]

# Treinar o modelo final com o melhor n_prototypes
lvq_best_user = LVQ1(n_prototypes=best_n_prototypes_val, learning_rate=0.1, n_epochs=100, random_state=42)
lvq_best_user.fit(X_train_user_lvq, y_train_user_lvq, X_val=X_test_user_lvq, y_val=y_test_user_lvq)
y_pred_user_lvq = lvq_best_user.predict(X_test_user_lvq)

# ─────────────────────────────────────────────────────────────────────
# Visualização dos Resultados para o Dataset do Usuário (LVQ1)
# ─────────────────────────────────────────────────────────────────────

fig_lvq_user = plt.figure(figsize=(17, 5))
gs_lvq_user = gridspec.GridSpec(1, 3, figure=fig_lvq_user, wspace=0.38)
fig_lvq_user.suptitle('LVQ1 — Análise com Dataset Pré-processado do Usuário',
                      fontsize=14, fontweight='bold', color='white', y=1.02)

# Painel 1 — Curva de acurácia vs n_prototypes
ax_lvq_user1 = fig_lvq_user.add_subplot(gs_lvq_user[0, 0])
ax_lvq_user1.plot(n_prototypes_range, acc_train_lvq, 'o-', color=COR_B, lw=2.5, ms=6, label='Treino')
ax_lvq_user1.plot(n_prototypes_range, acc_test_lvq,  's--', color=COR_A, lw=2.5, ms=6, label='Teste', alpha=0.85)
ax_lvq_user1.fill_between(n_prototypes_range, acc_train_lvq, acc_test_lvq,
                         where=[tr > te for tr, te in zip(acc_train_lvq, acc_test_lvq)],
                         alpha=0.15, color=COR_A, label='Gap overfitting')
ax_lvq_user1.axvline(x=best_n_prototypes_val, color='white', lw=1.5, ls=':', label=f'Melhor {best_n_prototypes_val} protótipos/classe')
ax_lvq_user1.set_xlabel('Nº de Protótipos por Classe'); ax_lvq_user1.set_ylabel('Acurácia')
ax_lvq_user1.set_ylim(0.5, 1.05); ax_lvq_user1.set_xticks(n_prototypes_range)
ax_lvq_user1.set_title(f'Acurácia vs Nº Protótipos (LVQ1)', fontweight='bold', color='white')
ax_lvq_user1.legend(facecolor='#1B2A4A', labelcolor='white', framealpha=0.9, fontsize=9)
ax_lvq_user1.grid(True)

# Painel 2 — Matriz de Confusão (melhor modelo LVQ1)
cm_lvq_user = confusion_matrix(y_test_user_lvq, y_pred_user_lvq, labels=user_class_names)

ax_lvq_user2 = fig_lvq_user.add_subplot(gs_lvq_user[0, 1])
im_lvq_user = ax_lvq_user2.imshow(cm_lvq_user, cmap='Oranges') # Using Oranges for LVQ to differentiate
ax_lvq_user2.set_xticks(range(len(user_class_names))); ax_lvq_user2.set_yticks(range(len(user_class_names)))
ax_lvq_user2.set_xticklabels(user_class_names, rotation=20, ha='right')
ax_lvq_user2.set_yticklabels(user_class_names)
ax_lvq_user2.set_xlabel('Predito'); ax_lvq_user2.set_ylabel('Real')
ax_lvq_user2.set_title(f'Matriz de Confusão (LVQ1 com {best_n_prototypes_val} p/classe)',
                       fontweight='bold', color='white')
for i in range(len(user_class_names)):
    for j in range(len(user_class_names)):
        ax_lvq_user2.text(j, i, str(cm_lvq_user[i, j]),
                         ha='center', va='center',
                         color='white' if cm_lvq_user[i, j] > cm_lvq_user.max()/2 else '#1B2A4A',
                         fontsize=14, fontweight='bold')

# Painel 3 — Relatório de Classificação (Texto)
ax_lvq_user3 = fig_lvq_user.add_subplot(gs_lvq_user[0, 2])
ax_lvq_user3.axis('off') # Esconder eixos para exibir texto
ax_lvq_user3.set_title('Relatório de Classificação (LVQ1)', fontweight='bold', color='white')

report_lvq_user_dict = classification_report(y_test_user_lvq, y_pred_user_lvq, target_names=user_class_names, output_dict=True)
report_lvq_str = """
{: <15} {: >8} {: >8} {: >8} {: >8}
""".format('Classe', 'Precision', 'Recall', 'F1-Score', 'Support')
report_lvq_str += "-" * 50 + "\n"

for label in user_class_names:
    metrics = report_lvq_user_dict.get(label, {})
    report_lvq_str += "{: <15} {: >8.2f} {: >8.2f} {: >8.2f} {: >8.0f}\n".format(
        label,
        metrics.get('precision', 0.0),
        metrics.get('recall', 0.0),
        metrics.get('f1-score', 0.0),
        metrics.get('support', 0)
    )

report_lvq_str += "\n" + "-" * 50 + "\n"
# Adicionar macro avg e weighted avg
for avg_type in ['accuracy', 'macro avg', 'weighted avg']:
    metrics = report_lvq_user_dict.get(avg_type, {})
    if isinstance(metrics, dict):
        report_lvq_str += "{: <15} {: >8.2f} {: >8.2f} {: >8.2f} {: >8.0f}\n".format(
            avg_type,
            metrics.get('precision', 0.0),
            metrics.get('recall', 0.0),
            metrics.get('f1-score', 0.0),
            metrics.get('support', 0)
        )
    else: # accuracy is not a dict
        report_lvq_str += "{: <15} {: >8.2f} {: >8} {: >8} {: >8.0f}\n".format(
            avg_type,
            metrics, '', '', report_lvq_user_dict['macro avg']['support'] # Total support for accuracy
        )

ax_lvq_user3.text(0, 1, report_lvq_str,
                  transform=ax_lvq_user3.transAxes,
                  fontsize=9, va='top', ha='left', color=TEXTO, family='monospace')

plt.tight_layout()
plt.show()

print("\n✅ LVQ1 com Dataset do Usuário concluído.")
print(f"   Melhor Nº de Protótipos por Classe encontrado: {best_n_prototypes_val}")
print(f"   Acurácia no Treino (melhor config): {lvq_best_user.score(X_train_user_lvq, y_train_user_lvq):.1%}")
print(f"   Acurácia no Teste (melhor config): {lvq_best_user.score(X_test_user_lvq, y_test_user_lvq):.1%}")


Iniciando integração LVQ1 com o dataset pré-processado do usuário...
✅ LVQ1 implementado.
Dataset do usuário — Treino: (81412, 23) | Teste: (20354, 23)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/tmp/ipykernel_18139/4145812466.py:209: UserWarni


✅ LVQ1 com Dataset do Usuário concluído.
   Melhor Nº de Protótipos por Classe encontrado: 1
   Acurácia no Treino (melhor config): 53.9%
   Acurácia no Teste (melhor config): 53.9%


IMPORTANDO MLFLOW


In [ ]:
# Instalar MLflow
!pip install -q mlflow

import mlflow
import mlflow.sklearn
import shutil, os
from mlflow.tracking import MlflowClient
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib
matplotlib.use('Agg')  # necessário no Colab para salvar figuras

EXPERIMENT_NAME = 'Diabetes_130_Hospitais'
print("MLflow importado com sucesso.")

MLflow importado com sucesso.


RESET DO EXPERIMENTO MLFLOW

In [ ]:
import shutil, os, mlflow, time
from mlflow.tracking import MlflowClient

# 1. Encerrar run ativa
if mlflow.active_run():
    mlflow.end_run()

# 2. Deletar tudo do disco
for path in ['mlruns', 'mlflow_diabetes130.db']:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
        print(f"'{path}' removido.")

# 3. Usar filesystem em vez de SQLite (evita problema de conexão presa)
mlflow.set_tracking_uri('file:///content/mlruns')

# 4. Nome único por sessão
EXPERIMENT_NAME = f'Diabetes_130_{int(time.time())}'
exp_id = mlflow.create_experiment(EXPERIMENT_NAME)
mlflow.set_experiment(experiment_id=exp_id)
print(f"\nExperimento '{EXPERIMENT_NAME}' criado (id={exp_id}). Pronto!")


Experimento 'Diabetes_130_1777499887' criado (id=184120296276657149). Pronto!


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Busca de Hiperparâmetros com MLflow (Árvore de Decisão e KNN)

In [ ]:
# Guard pattern
if mlflow.active_run():
    mlflow.end_run()

# Mapear labels para usar roc_auc_score em multiclasse
label_map = {label: i for i, label in enumerate(sorted(y_train.unique()))}
y_train_num = y_train.map(label_map)
y_test_num  = y_test.map(label_map)

def executar_experimento_diabetes(nome_modelo, estimador, param_dist):
    """Busca de hiperparâmetros com rastreamento MLflow adaptado para o dataset Diabetes 130."""
    if mlflow.active_run():
        mlflow.end_run()

    print(f"\n{'='*55}")
    print(f"  Iniciando busca: {nome_modelo}")
    print(f"{'='*55}")

    with mlflow.start_run(run_name=f'Busca_{nome_modelo}'):

        # 1. Busca com RandomizedSearchCV
        search = RandomizedSearchCV(
            estimador, param_dist,
            n_iter=10, cv=5, scoring='accuracy',
            random_state=42, n_jobs=-1
        )
        search.fit(X_train_final_df, y_train)

        # 2. Nested runs por tentativa
        for i in range(len(search.cv_results_['params'])):
            with mlflow.start_run(run_name=f'{nome_modelo}_Iter_{i}', nested=True):
                mlflow.log_params(search.cv_results_['params'][i])
                mlflow.log_metric('acc_validacao_cruzada',
                                  search.cv_results_['mean_test_score'][i])
                mlflow.set_tag('algoritmo', nome_modelo)

        # 3. Métricas no teste com o melhor modelo
        best = search.best_estimator_
        y_pred = best.predict(X_test_final_df)

        mlflow.log_params(search.best_params_)
        mlflow.log_metric('melhor_acc_cv', search.best_score_)
        mlflow.log_metric('acc_teste',       accuracy_score(y_test, y_pred))
        mlflow.log_metric('f1_teste',        f1_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric('precision_teste', precision_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric('recall_teste',    recall_score(y_test, y_pred, average='weighted'))
        mlflow.set_tag('algoritmo', nome_modelo)

        # 4. Matriz de confusão como artefato
        fig_cm, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
        ax.set_title(f'Matriz de Confusão — {nome_modelo}')
        plt.tight_layout()
        mlflow.log_figure(fig_cm, f'confusion_matrix_{nome_modelo}.png')
        plt.show(); plt.close()

        mlflow.sklearn.log_model(best, f'modelo_{nome_modelo}')

        print(f"  Melhor config: {search.best_params_}")
        print(f"  Acc CV: {search.best_score_:.3f} | Acc Teste: {accuracy_score(y_test, y_pred):.3f}")
        print(f"  F1 (weighted): {f1_score(y_test, y_pred, average='weighted'):.3f}")
        return best


# Executar para Árvore de Decisão
best_dt_mlflow = executar_experimento_diabetes(
    'DecisionTree',
    DecisionTreeClassifier(),
    {'max_depth': [3, 5, 8, 12, None], 'criterion': ['gini', 'entropy']}
)

# Executar para KNN
best_knn_mlflow = executar_experimento_diabetes(
    'KNN',
    KNeighborsClassifier(),
    {'n_neighbors': list(range(3, 25, 2)), 'weights': ['uniform', 'distance']}
)

print("\nBusca com MLflow concluída!")


  Iniciando busca: DecisionTree


2026/04/29 22:00:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 22:00:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  Melhor config: {'max_depth': 5, 'criterion': 'entropy'}
  Acc CV: 0.563 | Acc Teste: 0.563
  F1 (weighted): 0.483

  Iniciando busca: KNN


2026/04/29 22:06:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/29 22:06:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  Melhor config: {'weights': 'uniform', 'n_neighbors': 23}
  Acc CV: 0.551 | Acc Teste: 0.555
  F1 (weighted): 0.508

Busca com MLflow concluída!


 Comparação Final e Análise MLflow

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

runs_busca = runs[
    runs['tags.mlflow.runName'].str.startswith('Busca_', na=False)
].dropna(subset=['metrics.f1_teste', 'metrics.acc_teste'])

print("Comparação dos modelos (resultados rastreados no MLflow):\n")
cols_disp = [c for c in [
    'tags.algoritmo', 'metrics.acc_teste',
    'metrics.f1_teste', 'metrics.recall_teste',
    'metrics.precision_teste', 'metrics.melhor_acc_cv'
] if c in runs_busca.columns]

print(runs_busca[cols_disp].sort_values('metrics.f1_teste', ascending=False)
      .reset_index(drop=True).to_string())

# Identificar o campeão
melhor_run = runs_busca.loc[runs_busca['metrics.f1_teste'].idxmax()]
print(f"\nCampeão: {melhor_run['tags.algoritmo']}")
print(f"  F1 Teste:  {melhor_run['metrics.f1_teste']:.3f}")
print(f"  Acc Teste: {melhor_run['metrics.acc_teste']:.3f}")

Comparação dos modelos (resultados rastreados no MLflow):

  tags.algoritmo  metrics.acc_teste  metrics.f1_teste  metrics.recall_teste  metrics.precision_teste  metrics.melhor_acc_cv
0            KNN           0.554633          0.507842              0.554633                 0.509988               0.551356
1   DecisionTree           0.562592          0.483252              0.562592                 0.526732               0.563087

Campeão: KNN
  F1 Teste:  0.508
  Acc Teste: 0.555


Rede Neural MLP (adaptada da aula ANN)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)

# Encodar y para inteiros (resolve o TypeError com early_stopping)
le = LabelEncoder()
y_train_mlp = le.fit_transform(y_train.values)
y_test_mlp  = le.transform(y_test.values)
X_train_mlp = X_train_final_df.values
X_test_mlp  = X_test_final_df.values

print(f"Classes: {le.classes_} → {list(range(len(le.classes_)))}")
print("Iniciando busca de hiperparâmetros da MLP...\n")

activations  = ['relu', 'tanh', 'logistic']
hidden_sizes = [10, 25, 50, 75, 100]

k_scores_cv    = []
k_scores_train = []
k_scores_test  = []
config_labels  = []

for act in activations:
    for k in hidden_sizes:
        clf_cv = MLPClassifier(
            hidden_layer_sizes=(k, k), activation=act,
            early_stopping=False, random_state=42, max_iter=200
        )

        scores = cross_val_score(clf_cv, X_train_mlp, y_train_mlp, cv=5, scoring='accuracy')
        k_scores_cv.append(scores.mean())

        clf_fit = MLPClassifier(
            hidden_layer_sizes=(k, k), activation=act,
            early_stopping=True, n_iter_no_change=5,
            random_state=42, max_iter=200
        )
        clf_fit.fit(X_train_mlp, y_train_mlp)
        k_scores_train.append(clf_fit.score(X_train_mlp, y_train_mlp))
        k_scores_test.append(clf_fit.score(X_test_mlp, y_test_mlp))
        config_labels.append(f"{act[:3]}-{k}")
        print(f"  {act:<8} k={k:<4} | CV={scores.mean():.3f} | Treino={k_scores_train[-1]:.3f} | Teste={k_scores_test[-1]:.3f}")

# Plot
the_len = len(hidden_sizes)
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(range(len(k_scores_cv)),    k_scores_cv,    label='Validação Cruzada')
ax.plot(range(len(k_scores_train)), k_scores_train, label='Treinamento')
ax.plot(range(len(k_scores_test)),  k_scores_test,  label='Teste')

for i, (act, cor) in enumerate(zip(activations, ['gainsboro', 'bisque', 'aquamarine'])):
    ax.axvspan(i * the_len, (i+1) * the_len, color=cor, alpha=0.4)
    ax.annotate(act.capitalize(), xy=((i + 0.5) * the_len, max(k_scores_cv) + 0.005),
                ha='center', fontsize=11, fontweight='bold')

ax.set_xticks(range(len(config_labels)))
ax.set_xticklabels(config_labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Configurações (ativação - neurônios/camada)')
ax.set_ylabel('Acurácia')
ax.set_title('Busca MLP: Acurácia por Configuração')
ax.legend()
plt.tight_layout()
plt.show()

best_idx = int(np.argmax(k_scores_test))
best_act = activations[best_idx // the_len]
best_k   = hidden_sizes[best_idx % the_len]
print(f"\nMelhor configuração: ativação={best_act}, neurônios/camada={best_k}")
print(f"  CV={k_scores_cv[best_idx]:.3f} | Treino={k_scores_train[best_idx]:.3f} | Teste={k_scores_test[best_idx]:.3f}")

Classes: ['<30' '>30' 'NO'] → [0, 1, 2]
Iniciando busca de hiperparâmetros da MLP...

  relu     k=10   | CV=0.574 | Treino=0.573 | Teste=0.572
  relu     k=25   | CV=0.569 | Treino=0.576 | Teste=0.571
  relu     k=50   | CV=0.552 | Treino=0.583 | Teste=0.573
  relu     k=75   | CV=0.535 | Treino=0.583 | Teste=0.575
  relu     k=100  | CV=0.513 | Treino=0.581 | Teste=0.576
  tanh     k=10   | CV=0.576 | Treino=0.572 | Teste=0.569
  tanh     k=25   | CV=0.569 | Treino=0.570 | Teste=0.569
  tanh     k=50   | CV=0.550 | Treino=0.581 | Teste=0.577
  tanh     k=75   | CV=0.533 | Treino=0.579 | Teste=0.573
